# Homework 07: Integer programming models and logic

**ISyE 524 — Fall 2026**  
**Due:** Monday, November 2, 2026, at 11:59 p.m. (Madison time).

Complete all three problems. Model on/off operating ranges, logical
commitments, and minimum set covering. Write your mathematical formulation
before implementing it in Julia/JuMP. Use HiGHS and check solver status and
solution availability before reading values. Decision-variable definitions
are supplied; formulate the objectives and constraints and explain your reasoning.

Submit **one PDF to Gradescope** and mark which pages belong to each problem.
**You will not receive full marks unless you mark the pages.**


## Getting started and written answers

Follow [Assignments and PDF Submission](https://github.com/jlinderoth/isye524-students-julia/blob/main/docs/canvas-assignment-workflow.md). In VS Code, use **Tasks: Run Task** to run **ISyE 524: Update course repository**, then **ISyE 524: Start an assignment from its template**, entering `hw07`. Open `student-work/hw07/hw07.ipynb`. An existing working copy will not be overwritten.

For **every written part**, type LaTeX mathematics in Markdown or insert a clear photograph/scan of handwritten work. Include the answer and reasoning together and label the subparts; no duplicate transcription is required. Julia/JuMP implementations belong in code cells.

Use `$x_1 \geq 0$` for inline mathematics and `$$x_1+x_2\leq5$$` for display mathematics. Run the Markdown cell to render it.

For handwritten work, edit a Markdown cell in VS Code, drag in a cropped PNG/JPEG, and choose **Insert Image as Attachment**. Alternatively, put the image beside your working notebook under `student-work/hw07/images/` and insert `![My answer](images/problem1.png)`. Keep external images with the notebook. See [Images and handwritten work](https://github.com/jlinderoth/isye524-students-julia/blob/main/docs/assignments.md#images-and-handwritten-work).

Back up `student-work/hw07/` separately; Git does not back up your answers or images. After completing the notebook, export it with the course PDF task, inspect `submissions/hw07.pdf` for readable mathematics, tables, outputs, and images, and submit that PDF to Gradescope.


## Computational setup

Use the course Julia environment. All numerical data are supplied below;
no separate data files are required. Complete the model and reporting cells.
The supplied setup and data cells run without unfinished answers.

AI tools may help debug your own code, explain syntax or error messages,
and critique your work. They should not replace constructing your formulation,
implementation, or written solution. Complete the required collaboration
and LLM-use statement at the end.


In [ ]:
using JuMP, HiGHS, Printf
import MathOptInterface as MOI


## 1: Steam power with operating minimums

A power plant has three boilers feeding a common steam header and three
turbines that draw from it. In one planning period, each boiler and turbine
is either off or operates between its stated minimum and maximum. Steam
from any boiler may supply any turbine. Steam may be left unused.
There are no fixed startup costs.

| Boiler $i$ | Minimum steam if on (tons) | Maximum steam (tons) | Steam cost ($/ton) |
| :--- | ---: | ---: | ---: |
| B1 | 400 | 1,000 | 10 |
| B2 | 200 | 900 | 8 |
| B3 | 300 | 800 | 7 |

| Turbine $j$ | Minimum steam if on (tons) | Maximum steam (tons) | Electricity (kWh/ton) | Processing cost ($/ton) |
| :--- | ---: | ---: | ---: | ---: |
| T1 | 300 | 600 | 4 | 2 |
| T2 | 500 | 800 | 5 | 3 |
| T3 | 600 | 900 | 6 | 4 |

Use indices $i\in B=\{1,2,3\}$ for boilers and $j\in T=\{1,2,3\}$
for turbines, with these **supplied decision variables**:

- $x_i\ge0$: continuous tons of steam produced by boiler $i$.
- $t_j\ge0$: continuous tons of steam processed by turbine $j$.
- $y_i\in\{0,1\}$: 1 if boiler $i$ operates, 0 otherwise.
- $z_j\in\{0,1\}$: 1 if turbine $j$ operates, 0 otherwise.

### 1(a): Formulate and solve

The plant must produce **at least 8,000 kWh**. Formulate a mixed-integer
linear program minimizing total boiler and turbine costs. Include constraints
that connect each device's operating status to its steam quantity, account
for available steam, and meet the electricity requirement. Explain why the
operating-range constraints allow a device to be off.

Implement and solve your model. Report the operating devices, steam produced
by each boiler, steam processed by each turbine, total electricity, and total
cost, with units. Use `x[i]`, `t[j]`, `y[i]`, and `z[j]` for the variables.


**Your formulation and reasoning:**

[Type your answer here, or insert a clear photograph/scan of handwritten work.]


In [ ]:
B = 1:3
T = 1:3
L = Dict(zip(B, [400.0, 200.0, 300.0]))
U = Dict(zip(B, [1000.0, 900.0, 800.0]))
c = Dict(zip(B, [10.0, 8.0, 7.0]))
ell = Dict(zip(T, [300.0, 500.0, 600.0]))
u = Dict(zip(T, [600.0, 800.0, 900.0]))
q = Dict(zip(T, [4.0, 5.0, 6.0]))
k = Dict(zip(T, [2.0, 3.0, 4.0]));


In [ ]:
# Build and solve the model for 8,000 kWh.
# You may write a function to reuse the model in 1(b) and 1(c).
# Report device quantities, total electricity, and cost.


**Your reported results:**

[Type your answer here, or insert a clear photograph/scan of handwritten work.]


### 1(b): Lower electricity demand

Demand falls to **2,000 kWh**. Keep all operating minimums and maximums.
Solve the model again. Report only the operating devices and their steam
quantities, electricity, and total cost. Explain the change in turbine choice
from 1(a).


In [ ]:
# Solve the same operating-range model for 2,000 kWh and report
# the plan.


**Your answer and reasoning:**

[Type your answer here, or insert a clear photograph/scan of handwritten work.]


### 1(c): What if operating minimums are ignored?

Keep demand at **2,000 kWh**. Suppose devices may now operate at any
nonnegative level up to their maximum when enabled. Remove only the
operating-minimum constraints, keeping all other constraints and costs.
Solve again. Report the positive steam quantities, electricity, and cost.
Compare the cost with 1(b), and identify the operating requirement violated
by this plan under the original rules.


In [ ]:
# Solve the 2,000-kWh case with only the operating minimums
# removed.


**Your answer and reasoning:**

[Type your answer here, or insert a clear photograph/scan of handwritten work.]


## 2: Community projects with logical commitments

A community organization has **$24,000** for projects. Each project must
be undertaken completely or omitted. Costs below are in **thousands of
dollars**. Impact points are a planning score that adds across projects.
Field-team hours are the total hours required by each selected project.

| Project $p$ | Description | Cost $c_p$ | Impact $b_p$ | Hours $h_p$ |
| :--- | :--- | ---: | ---: | ---: |
| A | Community garden | 7 | 13 | 6 |
| B | Food-truck pop-up | 6 | 12 | 5 |
| C | Volunteer training | 5 | 9 | 4 |
| D | Tool library | 4 | 8 | 4 |
| E | Storage upgrade | 3 | 5 | 2 |
| F | Compost hub | 4 | 7 | 3 |
| G | Soil workshop | 2 | 4 | 1 |
| H | Nutrition class | 5 | 10 | 5 |

The organization has five operating commitments:

1. The garden requires volunteer training or the tool library, or both.
2. The food-truck pop-up and tool library cannot both be undertaken because
   they need the same trailer.
3. If both volunteer training and the tool library are undertaken, the
   storage upgrade must also be undertaken. The upgrade may be selected
   independently.
4. The compost hub and soil workshop must be undertaken together or both
   omitted.
5. Exactly one of the food-truck pop-up and nutrition class must be selected.

Selecting both the garden and nutrition class earns **four additional
impact points** for their combined program. This bonus is awarded once,
and only when both are selected. There is no other limit on total hours
in parts (a)–(b).

Use these **supplied decision variables**:

- $x_p\in\{0,1\}$ for $p\in P=\{A,B,C,D,E,F,G,H\}$: 1 if project $p$
  is selected, 0 otherwise.
- $w\in\{0,1\}$: 1 if and only if both A and H are selected.

### 2(a): Formulate the logical selection model

Formulate a binary linear program maximizing total impact, including the
joint-program bonus, subject to the budget and all five commitments.
Ensure the constraints give $w$ its stated meaning in every feasible
solution. Label the constraints corresponding to commitments 1–5.

Explain why commitment 1 permits both supporting projects and why commitment
3 permits the storage upgrade even when the training and tool library are
not both selected. Do not use products of decision variables.


**Your formulation and reasoning:**

[Type your answer here, or insert a clear photograph/scan of handwritten work.]


In [ ]:
P = [:A, :B, :C, :D, :E, :F, :G, :H]
c_project = Dict(zip(P, [7, 6, 5, 4, 3, 4, 2, 5]))
b_project = Dict(zip(P, [13, 12, 9, 8, 5, 7, 4, 10]))
h_project = Dict(zip(P, [6, 5, 4, 4, 2, 3, 1, 5]))
budget = 24;  # thousands of dollars


### 2(b): Solve and interpret

Implement and solve the model from 2(a). Use `x[p]` and `w` for the
variables. Report selected projects, project spending in dollars, total
field-team hours, and total impact including any bonus. State which
projects earn the bonus, if it is awarded.


In [ ]:
# Build and solve the project-selection model.
# Report selected projects, spending, hours, and impact.


**Your results and interpretation:**

[Type your answer here, or insert a clear photograph/scan of handwritten work.]


### 2(c): A workload-triggered supervisor fee

Now a supervisor must be hired if total field-team hours are **17 or more**.
Below 17 hours, no supervisor is hired. The supervisor costs **$3,000**,
which counts against the **same $24,000 budget**; it adds no impact points.
Keep all five operating commitments and the joint-program bonus.

Use the additional **supplied binary variable** $v$, equal to 1 if and
only if total field-team hours are at least 17. Write the added or changed
linear constraints and justify finite constants using the data. Explain
why the integer hour total permits an exact distinction between fewer
than 17 hours and at least 17 hours.

Implement and solve the revised model, using `v` for the new variable.
Report selected projects, project spending, supervisor cost, combined
spending, hours, and impact. Explain the change from 2(b).


**Your formulation and reasoning:**

[Type your answer here, or insert a clear photograph/scan of handwritten work.]


In [ ]:
# Add the exact workload indicator and include the supervisor in
# the budget.
# Solve and report the revised selection, costs, hours, and
# impact.


**Your results and explanation:**

[Type your answer here, or insert a clear photograph/scan of handwritten work.]


## 3: Auror posts and minimum set covering

The wizarding world has eight districts. The Ministry of Magic wants to
station auror posts so every district can be reached from at least one
post within **two seconds**. A post can protect its own district.
There is at most one post per district, and all posts have the same cost.
There is no capacity limit: one post can protect several districts.

For each district $i$, the table gives $N_i$, the candidate post locations
that can protect it. These coverage sets are supplied; no travel-time
calculation is required.

| District $i$ | Eligible post locations $N_i$ |
| :--- | :--- |
| 1 | 1, 5 |
| 2 | 2, 7 |
| 3 | 3, 4, 5 |
| 4 | 3, 4, 6 |
| 5 | 1, 3, 5, 6, 7 |
| 6 | 4, 5, 6, 8 |
| 7 | 2, 5, 7, 8 |
| 8 | 6, 7, 8 |

Use the **supplied decision variable** $x_j\in\{0,1\}$ for
$j\in J=\{1,\ldots,8\}$, equal to 1 if a post is opened at district
$j$, and 0 otherwise.

### 3(a): Formulate the covering model

Formulate an integer linear program minimizing the number of posts while
protecting all eight districts. Explain why the coverage requirement for
a district is an inequality rather than an equality.


**Your formulation and reasoning:**

[Type your answer here, or insert a clear photograph/scan of handwritten work.]


In [ ]:
I = 1:8
J = 1:8
N = Dict(1 => [1, 5],
         2 => [2, 7],
         3 => [3, 4, 5],
         4 => [3, 4, 6],
         5 => [1, 3, 5, 6, 7],
         6 => [4, 5, 6, 8],
         7 => [2, 5, 7, 8],
         8 => [6, 7, 8]);


### 3(b): Locate posts and verify coverage

Implement and solve your model, using `x[j]` for the binary decisions.
Report the minimum number of posts and their locations. For every district,
list the selected posts that protect it, verifying that none is unprotected.
Any optimal set of locations is acceptable. No column generation or routing
algorithm is required.


In [ ]:
# Build and solve the minimum set-covering model.
# Report open locations and, for each district, the open posts
# protecting it.


**Your results and interpretation:**

[Type your answer here, or insert a clear photograph/scan of handwritten work.]


## Collaboration and LLM-use statement — required

Complete this statement as the **very last part of your homework**. Be specific enough that a reader can understand what work you did yourself and how others or LLMs helped you learn.

- **People:** Give the name of every person you collaborated with, identify the problem/subpart you discussed, and explain the nature of the collaboration. If you worked alone, explicitly state that you did not collaborate with anyone.
- **LLMs/tools:** Name every LLM or AI assistant you used and identify the model/version as precisely as available. If the interface did not show a model/version, say so. If you used none, explicitly state that you did not use an LLM or AI assistant.
- **Specific assistance and learning:** For each tool, identify the problem/subpart, describe what you asked it to help with, what suggestions you used or rejected, and how the interaction helped you understand the mathematics or implementation. Explain how you checked the suggestions. “I used ChatGPT for help” is not sufficiently specific.

Disclosure does not replace the requirement that your submitted formulation, implementation, and explanation be your own work and that you understand them.


**People and collaboration:**

[Names, problem/subparts, and what you discussed; or explicitly state that you worked alone.]

**LLMs/tools and models:**

[Tool names and models/versions, or explicitly state that you used none.]

**How I used the assistance, what I learned, and how I checked it:**

[Specific details for each tool and relevant problem/subpart; state not applicable if you used none.]
